# Part 3: Comment Clustering with HDBSCAN
**Objective:** Group semantically similar Twitter comments using clustering techniques to enable structured analysis with an LLM. 

**Description:** Once the text embeddings are generated, clustering is performed using HDBSCAN to identify groups of semantically similar comments. This step helps organize large volumes of Twitter feedback into coherent clusters representing common themes or issues discussed by users. By grouping related comments together, the pipeline reduces noise and enables more efficient downstream analysis. Each cluster is then passed to the next stage of the system, when an LLM analyzes the grouped comments to extract insights and generate structured metrics about customer complaints and emerging trends. 

In [ ]:
!pip install umap-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import HDBSCAN
import umap
import os

In [ ]:
# Load embeddings file 
embeddings = np.load("tweet_embeddings.npy")
embeddings

# Load tweets file 
df = pd.read_parquet("tweets_clean.parquet")

In [ ]:
embeddings.shape

For the clustering part, we are going to apply HDBSCAN because this technique detects how many clusters exist without delimiting an exact number, which is exactly what we want for this project: to detect different complaints that group into the same topic. 

Another reason for using this technique is that HDBSCAN can assign a -1 to noisy comments, which could be non-related to some other topics that we want to obtain. 

Before applying this technique, we are going to apply UMAP. This is because embeddings have a high dimensionality and if we don't use a reduction dimension technique, we could provoke the problem of "Curse of Dimnesionality" in which all distances start to seem equal. 

In [ ]:
# Reduce embedding dimension
reducer = umap.UMAP(
    n_components = 50, 
    n_neighbors = 50, 
    min_dist = 0.05,
    metric = "cosine",
    random_state = 42
)

text_reduced = reducer.fit_transform(embeddings)

# HDBSCAN Clustering
clusterer = HDBSCAN(
    min_cluster_size = 30, 
    min_samples = 10, 
    metric = "euclidean"
)

clusters = clusterer.fit_predict(text_reduced)

In [ ]:
# Check the number of clusters
values, counts = np.unique(clusters, return_counts=True)
percentages = (counts / counts.sum() )* 100
print(values)
print(counts)
print(percentages)

As we can see the clustering has detected some outliers, however, the amount of noise is not even the 50% of the number of samples. And, since we are making topic discovery, this clusters can remain.

In [ ]:
# UMAP to visualize
viz_reducer = umap.UMAP(
    n_components=2,
    n_neighbors=50,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

text_2d = viz_reducer.fit_transform(embeddings)

# plot clusters 
plt.scatter(text_2d[:,0], text_2d[:,1], c=clusters, cmap="viridis", s=10)
plt.legend()
plt.title("HDBSCAN Clusters Visualization")
plt.show()

As we can see the clustering seems to be alright. We obtained: 
- compact clusters
- well separated clusters
- small visual noise
- small percentage of -1 cluster. 

In [ ]:
df["cluster"] = clusters

for cluster in np.unique(clusters): 
    print()
    print(cluster)
    print(df[df.cluster==cluster]["text"].sample(10))


In [ ]:
pd.crosstab(df["cluster"], df["airline"], normalize="index")

As we can see each cluster in its majority, are comments about the same airline. This is very insightful to the clusters, because the type of airline was not injected into the model, which means that the other comments that are captured within a majority of an airline, are because they might have a similar behavior than the others. 

In [ ]:
# Save Changes
path = "tweets_clusters.parquet"

df.to_parquet(path, index=False)

print("Saved:", os.path.exists(path))
print("Absolute path:", os.path.abspath(path))